# Proyecto Final IIC3633: Baselines

Alumnos:

* Alonso Casanova
* Bruno Cerda

Los algoritmos que elegimos de baselines para evaluarlos en la benchmark "PixelRec50k" son:

* Random
* Most Popular
* Alternating Least Squares (ALS)

## 1. Librerias

In [1]:
import gc
import random
import implicit
import pandas as pd
import numpy as np
import scipy.sparse as sparse
from typing import Union
from tqdm import tqdm

/home/brubuntu/Desktop/PUC/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
np.random.seed(5)

## 2. Datos

Los datos pueden ser descargados de este [link](https://drive.google.com/drive/folders/1bQPgM-6yAnzcD0jKBoUUheA9LL5xnCHG).

In [3]:
interactions = pd.read_csv("Datos/Pixel50k/interaction.csv")
item_info    = pd.read_csv("Datos/Pixel50k/item_info.csv")

In [4]:
def split_users(unique_uid: pd.Index, test_users_size: Union[float, int]) :

    n_users = unique_uid.size

    if isinstance(test_users_size, int):
        n_heldout_users = test_users_size
    else:
        n_heldout_users = int(test_users_size * n_users)

    tr_users = unique_uid[: (n_users - n_heldout_users * 2)]
    vd_users = unique_uid[(n_users - n_heldout_users * 2) : (n_users - n_heldout_users)]
    te_users = unique_uid[(n_users - n_heldout_users) :]

    return tr_users, vd_users, te_users

unique_uid = interactions.index
idx_perm = np.random.permutation(unique_uid.size)
unique_uid = unique_uid[idx_perm]
tr_users, vd_users, te_users = split_users(unique_uid, test_users_size=0.1)

train_df = interactions.loc[tr_users]
val_df   = interactions.loc[vd_users]
test_df  = interactions.loc[te_users]

## 3. Baselines

In [5]:
N = 10

### 3.1 Random

In [6]:
def recomendaciones_random(train_df, test_df, N):
    todos_los_items = set(train_df["item_id"].unique())

    vistos_train_dict = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    vistos_test_dict = test_df.groupby('user_id')['item_id'].apply(list).to_dict()

    usuarios_test = list(vistos_test_dict.keys())
    
    y_true = []
    y_pred = []
    for user_id in usuarios_test:
        vistos_train = vistos_train_dict.get(user_id, set())
        vistos_test = vistos_test_dict.get(user_id, [])[:N]
        
        candidatos = list(todos_los_items - vistos_train)
        random_items = random.sample(candidatos, N)

        y_true.append(vistos_test)
        y_pred.append(random_items)
        
    return y_true, y_pred

In [7]:
y_true_random, y_pred_random = recomendaciones_random(train_df, test_df, N)

### 3.2 Most Popular

In [8]:
def recomendaciones_most_popular(train_df, test_df, N):
    items_mas_populares = train_df.groupby('item_id').size().sort_values(ascending=False).index.tolist()

    vistos_train_dict = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    vistos_test_dict = test_df.groupby('user_id')['item_id'].apply(list).to_dict()

    usuarios_test = list(vistos_test_dict.keys())

    y_true = []
    y_pred = []
    for user_id in usuarios_test:
        vistos_train = vistos_train_dict.get(user_id, set())
        vistos_test = vistos_test_dict.get(user_id, []) 
        
        recomendaciones = []
        for item in items_mas_populares:
            if item not in vistos_train:
                recomendaciones.append(item)
            if len(recomendaciones) == N:
                break

        y_true.append(vistos_test)
        y_pred.append(recomendaciones)

    return y_true, y_pred

In [9]:
y_true_mostPop, y_pred_mostPop = recomendaciones_most_popular(train_df, test_df, N)

### 3.3 Alternating Least Squares (ALS)

In [10]:
user_id_map = dict(enumerate(train_df['user_id'].astype("category").cat.categories))
item_id_map = dict(enumerate(train_df['item_id'].astype("category").cat.categories))
user_to_idx = {real_id: internal_idx for internal_idx, real_id in user_id_map.items()}

In [11]:
train_df['user_cat'] = train_df['user_id'].astype("category").cat.codes
train_df['item_cat'] = train_df['item_id'].astype("category").cat.codes

sparse_item_user = sparse.csr_matrix(
    (np.ones(len(train_df)), (train_df['item_cat'], train_df['user_cat']))
)
sparse_user_item = sparse_item_user.T.tocsr()

In [12]:
model = implicit.als.AlternatingLeastSquares(
    factors=50, 
    iterations=15, 
    regularization=0.01,
    random_state=0
)
model.fit(sparse_user_item)

100%|██████████████████████████████████████████| 15/15 [00:00<00:00, 220.76it/s]


In [13]:
usuarios_test = test_df['user_id'].unique()
y_pred_dict = {}

for user_id in tqdm(usuarios_test):
    if user_id in user_to_idx:
        user_index = user_to_idx[user_id]
        user_history_row = sparse_user_item[user_index]
        recommended_indices, scores = model.recommend(
            userid=user_index, 
            user_items=user_history_row,
            N=N
        )
        y_pred_dict[user_id] = [item_id_map[idx] for idx in recommended_indices]
        
        del recommended_indices
        del scores
        del user_history_row
        gc.collect()
    else:
        y_pred_dict[user_id] = []

100%|█████████████████████████████████████| 39144/39144 [25:23<00:00, 25.69it/s]


In [14]:
vistos_test_dict = test_df.groupby('user_id')['item_id'].apply(list).to_dict()
usuarios_test = test_df['user_id'].unique()

y_true = []
y_pred = []
for user_id in usuarios_test:
    y_true.append(vistos_test_dict.get(user_id, []))
    y_pred.append(y_pred_dict.get(user_id, []))

## 4. Evaluación de métricas


Las métricas a utilizar son:


* MAP@k
* nDCG@k
* Recall@k


Con k=10 y k=5.

In [15]:
# De práctico_métricas.ipynb
def precision_at_k(r, k):
    assert 1 <= k <= r.size
    return (np.asarray(r)[:k] != 0).mean()

def average_precision_at_k(r, k):
    r = np.asarray(r)
    score = 0.
    for i in range(min(k, r.size)):
        score += precision_at_k(r, i + 1)
    return score / k

def dcg_at_k(r, k):
    r = np.asarray(r)[:k]
    if r.size:
        return np.sum(np.subtract(np.power(2, r), 1) / np.log2(np.arange(2, r.size + 2)))
    return 0.

def idcg_at_k(k):
    return dcg_at_k(np.ones(k), k)

def ndcg_at_k(r, k, max_relevant):
    idcg = idcg_at_k(min(k, max_relevant))
    if not idcg:
        return 0.
    return dcg_at_k(r, k) / idcg

def recall_at_k(relevant_items, recommended_items, k):
    relevant_items = set(relevant_items)
    recommended_items = set(recommended_items[:k])
    intersection = relevant_items.intersection(recommended_items)
    recall = len(intersection) / len(relevant_items) if len(relevant_items) > 0 else 0
    return recall

def evaluate_model(y_true, y_pred, N):
    map_scores = 0.
    ndcg_scores = 0.
    recall_scores = 0.
    
    for true_items, pred_items in zip(y_true, y_pred):
        
        true_set = set(true_items)
        
        # el relevance vector
        r = [1 if item in true_set else 0 for item in pred_items]
        
        # MAP
        user_map = average_precision_at_k(r, N)
        map_scores += user_map
        # NDGC@N
        max_relevant = len(true_items)
        user_ndcg = ndcg_at_k(r, N, max_relevant)
        ndcg_scores += user_ndcg
        # Recall@N
        user_recall = recall_at_k(true_items, pred_items, N)
        recall_scores += user_recall
        
    cantidad = len(y_true)
    final_map = map_scores / cantidad
    final_ndcg = ndcg_scores / cantidad
    final_recall = recall_scores / cantidad
    
    print(f"Métricas de evaluación ranking (Top-{N}):")
    print(f"MAP@{N}:    {final_map:.5f}")
    print(f"nDCG@{N}:   {final_ndcg:.5f}")
    print(f"Recall@{N}: {final_recall:.5f}")
    
    return final_map, final_ndcg, final_recall

### 4.1.1 Random, @10

In [16]:
evaluate_model(y_true_random, y_pred_random, N)

Métricas de evaluación ranking (Top-10):
MAP@10:    0.00003
nDCG@10:   0.00008
Recall@10: 0.00016


(3.134640837737098e-05, 8.072155528549554e-05, 0.00015753797942639144)

### 4.1.2 Random, @5

In [17]:
evaluate_model(y_true_random, y_pred_random, 5)

Métricas de evaluación ranking (Top-5):
MAP@5:    0.00003
nDCG@5:   0.00005
Recall@5: 0.00006


(2.5887322024661082e-05, 4.7397076809031365e-05, 6.429252673887867e-05)

### 4.2.1 Most Popular, @10

In [18]:
evaluate_model(y_true_mostPop, y_pred_mostPop, N)

Métricas de evaluación ranking (Top-10):
MAP@10:    0.00035
nDCG@10:   0.00078
Recall@10: 0.00133


(0.00034606045565579584, 0.0007760627254732525, 0.001325249597973388)

### 4.2.2 Most Popular, @5

In [19]:
evaluate_model(y_true_mostPop, y_pred_mostPop, 5)

Métricas de evaluación ranking (Top-5):
MAP@5:    0.00035
nDCG@5:   0.00050
Recall@5: 0.00057


(0.00035433272021254846, 0.0004978347561885957, 0.0005693269038226118)

### 4.3.1 ALS, @10

In [20]:
final_map, final_ndcg, final_recall = evaluate_model(y_true, y_pred, N=N)

Métricas de evaluación ranking (Top-10):
MAP@10:    0.00194
nDCG@10:   0.00414
Recall@10: 0.00674


### 4.3.2 ALS, @5

In [21]:
final_map, final_ndcg, final_recall = evaluate_model(y_true, y_pred, N=5)

Métricas de evaluación ranking (Top-5):
MAP@5:    0.00202
nDCG@5:   0.00296
Recall@5: 0.00363
